In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('ipl_bowl_features.csv')

# Get all column names
columns = df.columns.tolist()

# Find the index of 'venue' column
venue_index = columns.index('venue')

# Columns to rename (all columns after 'venue')
columns_to_rename = columns[venue_index + 1:]

# Create a mapping dictionary for renaming
rename_dict = {col: f"{col}_bowl" for col in columns_to_rename}

# Rename the columns
df = df.rename(columns=rename_dict)

# Save the modified dataframe back to CSV
df.to_csv('ipl_bowl_features_renamed.csv', index=False)

print("Column names after 'venue' have been updated with '_bowl' suffix.")

Column names after 'venue' have been updated with '_bowl' suffix.


In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('ipl_bat_features.csv')

# Get all column names
columns = df.columns.tolist()

# Find the index of 'venue' column
venue_index = columns.index('venue')

# Columns to rename (all columns after 'venue')
columns_to_rename = columns[venue_index + 1:]

# Create a mapping dictionary for renaming
rename_dict = {col: f"{col}_bat" for col in columns_to_rename}

# Rename the columns
df = df.rename(columns=rename_dict)

# Save the modified dataframe back to CSV
df.to_csv('ipl_bat_features_renamed.csv', index=False)

print("Column names after 'venue' have been updated with '_bat' suffix.")

Column names after 'venue' have been updated with '_bat' suffix.


In [ ]:
import pandas as pd

# Load both datasets
bat_df = pd.read_csv('ipl_bat_features_renamed.csv')
bowl_df = pd.read_csv('ipl_bowl_features_renamed.csv')

# First, verify which columns actually exist in both datasets
print("Batting columns:", bat_df.columns.tolist())
print("\nBowling columns:", bowl_df.columns.tolist())

# Rename columns to common names
bat_df = bat_df.rename(columns={
    'batter_id': 'player_id',
    'batter': 'player',
    'season_bat': 'season'  # Rename season_bat to season
})

bowl_df = bowl_df.rename(columns={
    'bowler_id': 'player_id',
    'bowler': 'player',
    'season_bowl': 'season'  # Rename season_bowl to season
})

# Reorder columns to put season right after venue in both datasets
def reorder_columns(df):
    cols = df.columns.tolist()
    if 'season' in cols and 'venue' in cols:
        cols.remove('season')
        venue_index = cols.index('venue')
        cols.insert(venue_index + 1, 'season')
        return df[cols]
    return df

bat_df = reorder_columns(bat_df)
bowl_df = reorder_columns(bowl_df)

# Identify which common columns actually exist in both datasets
common_cols = ['date', 'match_id', 'player_id', 'player', 'venue', 'season', 'inns', 'bat_team', 'bowl_team']
existing_common_cols = [col for col in common_cols
                       if col in bat_df.columns and col in bowl_df.columns]

print("\nCommon columns that exist in both datasets:", existing_common_cols)

# Merge the datasets using only the existing common columns
merged_df = pd.merge(
    bat_df,
    bowl_df,
    on=existing_common_cols,
    how='outer',
    suffixes=('_bat', '_bowl')
)

# Handle team columns - create single columns by combining _bat and _bowl versions
for team_col in ['bat_team', 'bowl_team']:
    if f"{team_col}_bat" in merged_df.columns and f"{team_col}_bowl" in merged_df.columns:
        merged_df[team_col] = merged_df[f"{team_col}_bat"].fillna(merged_df[f"{team_col}_bowl"])
        merged_df.drop([f"{team_col}_bat", f"{team_col}_bowl"], axis=1, inplace=True)

# Reorder columns to have common info first
player_cols = [col for col in existing_common_cols if col in merged_df.columns]
other_cols = [col for col in merged_df.columns if col not in player_cols]
merged_df = merged_df[player_cols + other_cols]

# Save the merged dataset
merged_df.to_csv('ipl_player_features_merged_final.csv', index=False)

print("\nMerge completed successfully!")
print(f"Final columns ({len(merged_df.columns)}):", merged_df.columns.tolist())
print("\nFirst 2 rows:")
print(merged_df.head(2))

Batting columns: ['date', 'match_id', 'bat_team', 'bowl_team', 'inns', 'batter_id', 'batter', 'venue', 'career_avg_bat', 'career_rpm_bat', 'career_sr_bat', 'career_bpm_bat', 'career_bfpm_bat', 'inns_avg_bat', 'inns_rpm_bat', 'inns_sr_bat', 'inns_bpm_bat', 'inns_bfpm_bat', 'ven_avg_bat', 'ven_rpm_bat', 'ven_sr_bat', 'ven_bpm_bat', 'ven_bfpm_bat', 'season_bat', 'ssn_avg_bat', 'ssn_rpm_bat', 'ssn_sr_bat', 'ssn_bpm_bat', 'ssn_bfpm_bat', 'last5_avg_bat', 'last5_sr_bat', 'last5_rpm_bat', 'last5_bfpm_bat', 'last5_bpm_bat']

Bowling columns: ['date', 'match_id', 'bowl_team', 'bat_team', 'bowler_id', 'bowler', 'inns', 'venue', 'season_bowl', 'career_avg_bowl', 'career_eco_bowl', 'career_sr_bowl', 'career_wpm_bowl', 'career_opm_bowl', 'inns_avg_bowl', 'inns_eco_bowl', 'inns_sr_bowl', 'inns_wpm_bowl', 'inns_opm_bowl', 'venue_avg_bowl', 'venue_eco_bowl', 'venue_sr_bowl', 'venue_wpm_bowl', 'venue_opm_bowl', 'ssn_avg_bowl', 'ssn_eco_bowl', 'ssn_sr_bowl', 'ssn_wpm_bowl', 'ssn_opm_bowl', 'last5_avg_bo

In [ ]:
import pandas as pd

# Load the merged dataset
merged_df = pd.read_csv('ipl_player_features_merged_final.csv')

# Identify columns to fill with 0 (all except these will remain unchanged)
columns_to_exclude = ['date', 'match_id', 'player_id', 'player', 'venue', 'bat_team', 'bowl_team']
columns_to_fill = [col for col in merged_df.columns if col not in columns_to_exclude]

# Fill specified columns with 0
merged_df[columns_to_fill] = merged_df[columns_to_fill].fillna(0)

# Save the cleaned dataset
merged_df.to_csv('ipl_player_features_merged_cleaned.csv', index=False)

print("Null values in performance metrics filled with 0")
print(f"Dataset saved to 'ipl_player_features_merged_cleaned.csv'")

Null values in performance metrics filled with 0
Dataset saved to 'ipl_player_features_merged_cleaned.csv'


In [ ]:
import pandas as pd

# Load your merged dataset
df = pd.read_csv('ipl_player_features_merged_cleaned.csv')

# Print all column names
print("All column names in the dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

# Print the number of columns
print(f"\nTotal columns: {len(df.columns)}")

# Optional: Print the first few rows to see sample data
print("\nSample data (first 2 rows):")
print(df.head(2))

All column names in the dataset:
1. date
2. match_id
3. player_id
4. player
5. venue
6. season
7. inns
8. bat_team
9. bowl_team
10. career_avg_bat
11. career_rpm_bat
12. career_sr_bat
13. career_bpm_bat
14. career_bfpm_bat
15. inns_avg_bat
16. inns_rpm_bat
17. inns_sr_bat
18. inns_bpm_bat
19. inns_bfpm_bat
20. ven_avg_bat
21. ven_rpm_bat
22. ven_sr_bat
23. ven_bpm_bat
24. ven_bfpm_bat
25. ssn_avg_bat
26. ssn_rpm_bat
27. ssn_sr_bat
28. ssn_bpm_bat
29. ssn_bfpm_bat
30. last5_avg_bat
31. last5_sr_bat
32. last5_rpm_bat
33. last5_bfpm_bat
34. last5_bpm_bat
35. career_avg_bowl
36. career_eco_bowl
37. career_sr_bowl
38. career_wpm_bowl
39. career_opm_bowl
40. inns_avg_bowl
41. inns_eco_bowl
42. inns_sr_bowl
43. inns_wpm_bowl
44. inns_opm_bowl
45. venue_avg_bowl
46. venue_eco_bowl
47. venue_sr_bowl
48. venue_wpm_bowl
49. venue_opm_bowl
50. ssn_avg_bowl
51. ssn_eco_bowl
52. ssn_sr_bowl
53. ssn_wpm_bowl
54. ssn_opm_bowl
55. last5_avg_bowl
56. last5_eco_bowl
57. last5_sr_bowl
58. last5_wpm_bowl
5

In [ ]:
import pandas as pd
from tqdm import tqdm  # For progress tracking

def enhanced_fantasy_mapping():
    # Load datasets with dtype specification for memory efficiency
    dtype_spec = {
        'match_id': 'str',
        'player_id': 'str',
        'player': 'str'
    }

    # Load datasets with error handling
    try:
        merged_df = pd.read_csv('ipl_player_features_merged_cleaned.csv', dtype=dtype_spec)
        fantasy_df = pd.read_csv('player_match_fantasy_points_with_dream_team.csv', dtype=dtype_spec)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return None

    # Data validation checks
    def validate_data(df, df_name):
        required_cols = {'match_id', 'player_id'}
        missing = required_cols - set(df.columns)
        if missing:
            print(f"Missing columns in {df_name}: {missing}")
            return False
        return True

    if not all([validate_data(merged_df, "merged data"),
               validate_data(fantasy_df, "fantasy data")]):
        return None

    # Create composite key for both datasets
    merged_df['composite_key'] = merged_df['match_id'] + '_' + merged_df['player_id']
    fantasy_df['composite_key'] = fantasy_df['match_id'] + '_' + fantasy_df['player_id']

    # Create lookup dictionary with progress bar
    print("\nCreating fantasy points lookup dictionary...")
    fantasy_lookup = {}
    for _, row in tqdm(fantasy_df.iterrows(), total=len(fantasy_df)):
        fantasy_lookup[row['composite_key']] = row.get('final_dt', None)

    # Map values with comprehensive checks
    print("\nMapping fantasy points to merged data...")
    merged_df['final_dt'] = None

    missing_count = 0
    for idx, row in tqdm(merged_df.iterrows(), total=len(merged_df)):
        key = row['composite_key']
        if key in fantasy_lookup:
            merged_df.at[idx, 'final_dt'] = fantasy_lookup[key]
        else:
            missing_count += 1

    # Post-mapping validation
    total_rows = len(merged_df)
    matched_rows = total_rows - missing_count
    match_percentage = (matched_rows / total_rows) * 100

    print(f"\nResults:")
    print(f"Total rows processed: {total_rows}")
    print(f"Successfully matched: {matched_rows} ({match_percentage:.2f}%)")
    print(f"Missing matches: {missing_count}")

    if missing_count > 0:
        # Create DataFrame of unmatched entries for debugging
        unmatched_df = merged_df[merged_df['final_dt'].isna()][['match_id', 'player_id', 'player']]
        print("\nSample of unmatched entries:")
        print(unmatched_df.head())

        # Save unmatched entries for review
        unmatched_df.to_csv('unmatched_entries.csv', index=False)
        print("Saved unmatched entries to 'unmatched_entries.csv'")

    # Save the enhanced dataset
    output_path = 'ipl_players_features_merged_enhanced.csv'
    merged_df.drop(columns=['composite_key'], inplace=True)
    merged_df.to_csv(output_path, index=False)

    print(f"\nEnhanced dataset saved to {output_path}")
    return merged_df

# Execute the function
final_df = enhanced_fantasy_mapping()
if final_df is not None:
    print("\nFirst 5 rows of final dataset:")
    print(final_df[['match_id', 'player_id', 'player', 'final_dt']].head())


Creating fantasy points lookup dictionary...


100%|██████████| 24701/24701 [00:01<00:00, 19635.98it/s]



Mapping fantasy points to merged data...


100%|██████████| 48777/48777 [00:03<00:00, 12429.58it/s]



Results:
Total rows processed: 48777
Successfully matched: 48777 (100.00%)
Missing matches: 0

Enhanced dataset saved to ipl_players_features_merged_enhanced.csv

First 5 rows of final dataset:
  match_id player_id          player final_dt
0   336001     44936  AB de Villiers        1
1   336004     44936  AB de Villiers        1
2   336009     44936  AB de Villiers        0
3   336015     44936  AB de Villiers        1
4   336017     44936  AB de Villiers        0


In [ ]:
!pip uninstall cuml -y
!pip uninstall pycaret -y
!pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.7/169.7 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 94.7 MB/s eta 0:00:00
   ━━━

In [ ]:
import pandas as pd
from pycaret.classification import *
from sklearn.preprocessing import LabelEncoder
import numpy as np
import joblib

# Load the enhanced dataset
df = pd.read_csv('ipl_players_features_merged_enhanced.csv')

# 1. Data Preprocessing ------------------------------------------------------

# Handle missing values in target
print(f"Initial missing values in target: {df['final_dt'].isna().sum()}")
df = df.dropna(subset=['final_dt'])

# Identify categorical columns (excluding IDs and target)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [col for col in categorical_cols
                   if col not in ['match_id', 'player_id', 'player', 'final_dt', 'date']]  # Added 'date' to exclusion

print(f"\nCategorical features to encode: {categorical_cols}")

# Columns to ignore/drop
cols_to_ignore = ['date', 'match_id', 'player']

# Verify which columns actually exist in the dataframe
existing_cols_to_ignore = [col for col in cols_to_ignore if col in df.columns]
print(f"\nColumns being ignored: {existing_cols_to_ignore}")

# Label Encoding for categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le  # Save encoders for inverse transform later

# 2. PyCaret Setup with Categorical Handling ---------------------------------

clf_setup = setup(
    data=df,
    target='final_dt',
    session_id=42,
    train_size=0.8,
    categorical_features=categorical_cols,  # Tell PyCaret these are categorical
    normalize=True,
    transformation=True,
    ignore_features=existing_cols_to_ignore,  # Only ignore columns that exist
    verbose=False
)

# 3. Model Training & Evaluation --------------------------------------------

# Compare models (sorted by Accuracy for classification)
best_model = compare_models(
    sort='Accuracy',
    n_select=3,
    fold=3,
    exclude=['dummy']
)

# Tune top models
tuned_models = []
for model in best_model:
    print(f"\nTuning {type(model).__name__}...")
    tuned = tune_model(
        model,
        optimize='Accuracy',
        n_iter=5,
        choose_better=True
    )
    tuned_models.append(tuned)

# Finalize best model
final_model = automl(optimize='Accuracy')

# 4. Save Artifacts --------------------------------------------------------

# Save model + encoders
save_model(final_model, 'ipl_fantasy_classifier')
joblib.dump(label_encoders, 'label_encoders.pkl')

# Generate predictions with inverse transformed categories
predictions = predict_model(final_model, data=df)

# Optional: Inverse transform categorical features for interpretability
for col in categorical_cols:
    if col in predictions.columns:
        predictions[col] = label_encoders[col].inverse_transform(predictions[col])

# Include only relevant columns in output
output_cols = ['player_id', 'final_dt', 'prediction_label'] + [col for col in categorical_cols if col in predictions.columns]
predictions[output_cols].to_csv('classification_predictions.csv', index=False)

print("\nTraining completed! Model and encoders saved.")
print(f"Final model type: {type(final_model).__name__}")

Initial missing values in target: 0

Categorical features to encode: ['venue', 'bat_team', 'bowl_team']

Columns being ignored: ['date', 'match_id', 'player']


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.6719,0.5989,0.9714,0.6780,0.7986,0.0473,0.0874,9.1100
gbc,Gradient Boosting Classifier,0.6714,0.5975,0.9932,0.6724,0.8019,0.0170,0.0567,18.7500
ada,Ada Boost Classifier,0.6695,0.5871,0.9883,0.6722,0.8001,0.0157,0.0450,6.0100
lr,Logistic Regression,0.6692,0.5633,0.9966,0.6701,0.8014,0.0036,0.0188,7.7467
ridge,Ridge Classifier,0.6692,0.5636,0.9971,0.6700,0.8014,0.0029,0.0169,2.9133
lda,Linear Discriminant Analysis,0.6691,0.5635,0.9956,0.6702,0.8011,0.0045,0.0211,3.0900
svm,SVM - Linear Kernel,0.6677,0.5238,0.9950,0.6695,0.8004,-0.0002,0.0009,4.0833
xgboost,Extreme Gradient Boosting,0.6578,0.5806,0.9137,0.6826,0.7814,0.0643,0.0819,4.3300
rf,Random Forest Classifier,0.6488,0.5580,0.8799,0.6851,0.7704,0.0710,0.0822,12.8000
et,Extra Trees Classifier,0.6353,0.5390,0.8526,0.6821,0.7579,0.0547,0.0607,10.4800


Processing:   0%|          | 0/63 [00:00<?, ?it/s]


Tuning LGBMClassifier...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6695,0.5883,0.9518,0.6812,0.7941,0.0621,0.0947
1,0.6676,0.5986,0.9464,0.6813,0.7922,0.0610,0.0902
2,0.6730,0.6045,0.9502,0.6842,0.7956,0.0765,0.1128
3,0.6725,0.6028,0.9529,0.6831,0.7958,0.0714,0.1079
4,0.6674,0.6078,0.9479,0.6806,0.7923,0.0590,0.0883
5,0.6717,0.5976,0.9544,0.6821,0.7956,0.0675,0.1037
6,0.6722,0.5960,0.9541,0.6826,0.7958,0.0698,0.1066
7,0.6715,0.6023,0.9537,0.6821,0.7953,0.0674,0.1031
8,0.6627,0.5694,0.9445,0.6781,0.7894,0.0460,0.0688


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

... (8 outputs left out)

Fitting 10 folds for each of 5 candidates, totalling 50 fits


Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
Transformation Pipeline and Model Successfully Saved


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.6945,0.7264,0.9868,0.6901,0.8122,0.1146,0.2078



Training completed! Model and encoders saved.
Final model type: LGBMClassifier
